# Notebook 02 — Electric Field and Streamlines

This notebook extends Notebook 01 by computing the electric field from a 2D electrostatic potential for a simple surface-style electrode layout.

It focuses on:

- solving Laplace's equation on a rectangular grid
- computing the electric field from the potential
- visualizing field magnitude
- plotting field streamlines
- examining a vertical centerline diagnostic

This provides the next step toward RF pseudopotential and trapping analysis.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Grid and electrode geometry

We reuse the same simple surface-style layout as Notebook 01:

- left DC electrode: negative voltage
- center RF-like electrode: positive voltage
- right DC electrode: negative voltage
- grounded outer boundaries


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Potential array and mask for fixed nodes
V = np.zeros((ny, nx), dtype=float)
fixed = np.zeros((ny, nx), dtype=bool)

# Ground outer boundaries
V[:, 0] = 0.0
V[:, -1] = 0.0
V[-1, :] = 0.0
fixed[:, 0] = True
fixed[:, -1] = True
fixed[-1, :] = True

# Lower boundary electrode pattern
bottom = 0
left_dc = (x >= -2.2) & (x <= -0.9)
center_rf = (x >= -0.5) & (x <= 0.5)
right_dc = (x >= 0.9) & (x <= 2.2)
remaining_ground = ~(left_dc | center_rf | right_dc)

V[bottom, left_dc] = -1.0
V[bottom, center_rf] = 1.0
V[bottom, right_dc] = -1.0
V[bottom, remaining_ground] = 0.0
fixed[bottom, :] = True

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')


## Solve Laplace equation

We solve the charge-free electrostatic equation:

$$
\nabla^2 V = 0
$$

using a standard 5-point finite-difference stencil and a sparse linear solve.


In [ ]:
def idx(i, j, nx):
    return i * nx + j

N = nx * ny
A = lil_matrix((N, N), dtype=float)
b = np.zeros(N, dtype=float)

cx = 1.0 / dx**2
cy = 1.0 / dy**2
c0 = -2.0 * (cx + cy)

for i in range(ny):
    for j in range(nx):
        k = idx(i, j, nx)

        if fixed[i, j]:
            A[k, k] = 1.0
            b[k] = V[i, j]
        else:
            A[k, idx(i, j, nx)] = c0
            A[k, idx(i, j - 1, nx)] = cx
            A[k, idx(i, j + 1, nx)] = cx
            A[k, idx(i - 1, j, nx)] = cy
            A[k, idx(i + 1, j, nx)] = cy

solution = spsolve(A.tocsr(), b)
Vsol = solution.reshape((ny, nx))

print('Laplace solve complete.')
print(f'Potential range: [{Vsol.min():.3f}, {Vsol.max():.3f}]')


## Compute electric field

The electric field is obtained from the potential via:

$$
\mathbf{E} = -\nabla V
$$

We compute the field components and the field magnitude.


In [ ]:
dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
Ex = -dV_dx
Ey = -dV_dy
E_mag = np.sqrt(Ex**2 + Ey**2)

print(f'Field magnitude range: [{E_mag.min():.3e}, {E_mag.max():.3e}]')


## Potential and field magnitude


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Vsol, shading='auto')
cs = ax.contour(X, Y, Vsol, levels=15)
ax.clabel(cs, inline=True, fontsize=8)

ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)

ax.set_title('2D Electrostatic Potential')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='Potential')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, E_mag, shading='auto')
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Electric Field Magnitude')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='|E|')
plt.tight_layout()
plt.show()


## Field streamlines

We plot streamlines of the electric field to visualize how the electrode layout shapes the field above the surface.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.pcolormesh(X, Y, E_mag, shading='auto', alpha=0.85)
ax.streamplot(x, y, Ex, Ey, density=1.1, linewidth=1.0)

ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=7)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=7)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=7)

ax.set_title('Electric Field Streamlines')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()


## Vertical centerline diagnostics

We inspect the potential and field magnitude along the vertical centerline at $x=0$.


In [ ]:
j0 = np.argmin(np.abs(x - 0.0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y, Vsol[:, j0], label='V(x=0, y)')
ax.set_title('Centerline Potential')
ax.set_xlabel('y')
ax.set_ylabel('Potential')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y, E_mag[:, j0], label='|E|(x=0, y)')
ax.set_title('Centerline Field Magnitude')
ax.set_xlabel('y')
ax.set_ylabel('|E|')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook adds field-level diagnostics to the basic electrostatic solver.

Natural next steps:

- compute an RF pseudopotential from $|E|^2$
- estimate curvature and candidate trapping height
- vary electrode gaps and widths
- sweep applied voltages
- compare field structure across geometries

That progression will move the workflow toward practical ion-trap analysis.
